# Classifying and clustering FIFA 21 players

Aadil Ali Vagh and Nicodemus Ong. October 2023.

This notebook predicts whether an outfield player in FIFA 21 is attacking or defensive, then groups players without using that label. The source is the public [FIFA 21 messy raw dataset](https://www.kaggle.com/datasets/yagunnersya/fifa-21-messy-raw-dataset-for-cleaning-exploring) scraped from SoFIFA: 18,979 players and 77 columns in `fifa21_raw_data.csv`.

**Result.** After dropping 2,075 goalkeepers, 8,004 players are attacking and 8,900 are defensive. A null model scores an AUC of 0.50. Interceptions alone reach an AUC of about 0.93. A decision tree and a 10-nearest-neighbours model both land near 0.91 accuracy on a 30% holdout. Four clusters separate a higher-rated defensive-leaning group, a strongly defensive group, an attacking group, and a younger group. The attacking cluster is not stable under bootstrap resampling.


## How to read this

The code is R from the original analysis. The plots are the saved figures, so the notebook can be read on GitHub without re-running it. Running the cells needs R and the Kaggle CSV, which is not in this folder.

1. Clean the SoFIFA export.
2. Define attacking and defensive, and compare with a null model.
3. Fit a decision tree and KNN on two feature sets.
4. Cluster players and check whether the groups hold up.
5. Link the dashboard and the steps to reproduce the work.


## 1. Cleaning the export

Club and contract expiry share one field. Money columns mix a euro sign with `K` and `M`. `Positions` lists several roles, height is in feet and inches, and the star ratings are glyphs. The snippets below are the conversions that mattered. Column names were later rewritten to lowercase snake case (`Value` became `value`, and so on).


In [ ]:
data <- "fifa21_raw_data.csv"
df <- read.csv(data, header = TRUE, na.strings = c("", "NA"))

# "€1.2M" / "€500K" -> numeric euros
symbols_to_zeroes <- function(value) {
  value <- gsub("€", "", value)
  if (grepl("M$", value)) {
    return(as.numeric(gsub("M", "e6", value)))
  } else if (grepl("K$", value)) {
    return(as.numeric(gsub("K", "e3", value)))
  } else {
    return(as.numeric(value))
  }
}

df$Value <- sapply(df$Value, symbols_to_zeroes)
df$Release.Clause <- sapply(df$Release.Clause, symbols_to_zeroes)

df$Wage <- str_remove_all(df$Wage, "€")
df$Wage <- str_replace_all(df$Wage, "K", "000")
df$Wage <- as.numeric(df$Wage)

# "FC Barcelona\n2004 ~ 2021" -> club and contract end
df$Club <- str_trim(str_extract(df$Team...Contract, "[[:alnum:]. ]+"))
df$ContractExpiry <- str_extract(df$Team...Contract, "(?<=~ )[^\n]+")

# Keep the first listed role as the player's position
df$Positions <- sapply(str_split(df$Positions, " "), "[", 1)


In [ ]:
# After the rename to snake_case
feet <- as.numeric(gsub("'.*", "", df$height))
inches <- as.numeric(gsub(".*'", "", gsub("\"", "", df$height)))
df$height <- feet * 12 + inches
df$weight <- as.numeric(gsub("lbs", "", df$weight))

df$weak_foot <- as.integer(gsub("★", "", df$weak_foot))
df$skill_moves <- as.integer(gsub("★", "", df$skill_moves))
df$international_reputation <- as.integer(gsub("★", "", df$international_reputation))


## 2. Attacking or defensive

Fullbacks, wing-backs, centre-backs, and defensive midfielders are defensive. Wingers, attacking midfielders, strikers, and centre-forwards are attacking. Central midfielders are split on the median defending rating among central midfielders. Goalkeepers are removed. CDM appears in both lists in the original code; `case_when` checks the defensive list first, so those players stay defensive.

That leaves 8,004 attacking and 8,900 defensive players. Numeric attributes were then cut into five bins. A 70/30 split with `set.seed(4009)` on that binned frame gives 11,832 training rows and 5,072 test rows.


In [ ]:
defensive_players <- c("LB", "LWB", "RB", "RWB", "CB", "CDM")
attacking_players <- c("LM", "RM", "CAM", "CDM", "RW", "ST", "LW", "CF")
median_defending <- median(df[df$Positions == "CM", "Defending"])

df <- df %>%
  mutate(
    PositionType = case_when(
      Positions %in% defensive_players ~ "Defensive Player",
      Positions %in% attacking_players ~ "Attacking Player",
      Positions == "GK" ~ "Goalkeeper",
      Positions == "CM" & Defending >= median_defending ~ "Defensive Player",
      Positions == "CM" & Defending < median_defending ~ "Attacking Player",
      TRUE ~ NA_character_
    )
  )

df <- df %>% filter(PositionType != "Goalkeeper")


In [ ]:
# cat_df: binned attributes, positiontype coded 1 = attacking and 0 = defensive
target <- "positiontype"
set.seed(4009)

train_size <- floor(0.7 * nrow(cat_df))
train_indices <- sample(seq_len(nrow(cat_df)), size = train_size)
train_set <- cat_df[train_indices, ]
test_set <- cat_df[-train_indices, ]

# Majority-rate baseline. AUC 0.50, MSE 0.25 on the test set.
null_model <- sum(train_set[[target]]) / nrow(train_set)


## 3. One attribute is already a strong signal

A single-variable model, scored with 100 resamples, puts interceptions at a mean AUC of 0.928 (SD 0.010). Aggression is about 0.77 and the physical summary about 0.72. The ROC curve is the same story: interceptions rise much faster than aggression.

![ROC curves for aggression and interceptions](figures/roc-single-variable.png)


## 4. Decision tree and KNN

Chi-square filtering (`FSelector::chi.squared`) keeps eight binned attributes with importance of at least 0.20:

| Attribute | Importance |
| --- | ---: |
| Interceptions | 0.414 |
| Aggression | 0.321 |
| Physical summary | 0.273 |
| Defensive work rate | 0.248 |
| Skill moves | 0.243 |
| Vision | 0.230 |
| Strength | 0.206 |
| Curve | 0.203 |

The tree is an `rpart` classifier on that formula. The root split is `interceptions_category >= 3`. Deeper leaves use a higher interceptions cut, then vision, skill moves, and aggression. Counts in each node are defensive, then attacking.


In [ ]:
chi_features <- c(
  "interceptions_category", "aggression_category", "summary_physical_category",
  "defensive_work_rate_category", "skill_moves_category", "vision_category",
  "strength_category", "curve_category"
)

filtered_train_set <- train_set[, !grepl("^pred", names(train_set))]
chi_sq_result <- chi.squared(positiontype ~ ., data = filtered_train_set)

formula_str <- paste("positiontype ~", paste(chi_features, collapse = " + "))
decision_model <- rpart(as.formula(formula_str), data = train_set, method = "class")
rpart.plot(decision_model, type = 4, extra = 1)


![Decision tree on the chi-square features](figures/decision-tree.png)

KNN uses the same chi-square columns, after centering and scaling the training set and applying that transformation to the test set. `k` is 10.


In [ ]:
normalization_values <- preProcess(train_set, method = c("center", "scale"))
train_set_normalized <- predict(normalization_values, train_set)
test_set_normalized <- predict(normalization_values, test_set)

train_predictors <- train_set_normalized[, chi_features]
test_predictors <- test_set_normalized[, chi_features]
train_response <- train_set_normalized[["positiontype"]]

k <- 10
knn_model <- knn(
  train = train_predictors,
  test = test_predictors,
  cl = train_response,
  k = k
)


### A second feature set from other FIFA papers

The same two models were fit again on eight attributes that come up repeatedly in published FIFA work: interceptions, dribbling, long passing, vision, ball control, acceleration, agility, and aggression.


In [ ]:
literature_review <- c(
  "interceptions_category", "dribbling_category", "long_passing_category",
  "vision_category", "ball_control_category", "acceleration_category",
  "agility_category", "aggression_category"
)

predictors <- paste(literature_review, collapse = " + ")
formula_str <- paste("positiontype ~", predictors)
decision_model_lr <- rpart(as.formula(formula_str), data = train_set, method = "class")


### What the holdout shows

Precision, recall, specificity, and F1 use the **defensive** class as the positive class. That is the encoding the confusion-matrix metrics line up with. Accuracy is the share of all test players labelled correctly.

On the chi-square features the models are close. The tree's recall of defensive players is 0.96, against 0.95 for KNN. F1 is 0.92 against 0.91.

On the literature features the split is clearer. The tree is more precise (0.93 against 0.88) and more specific (0.93 against 0.86). KNN recovers more of the defensive players (recall 0.93 against 0.89).

The green AUC bar is labeled 0.96 on the chi-square chart and 0.95 on the literature chart. Both are above the interceptions-only AUC of 0.93 and the null model at 0.50. The table is computed from the heatmap counts, rounded to two decimals.

![Chi-square feature comparison](figures/chi-square-comparison.png)

![Literature-feature comparison](figures/literature-comparison.png)

|  | Chi-square tree | Chi-square KNN | Literature tree | Literature KNN |
| --- | ---: | ---: | ---: | ---: |
| Accuracy | 0.91 | 0.91 | 0.91 | 0.90 |
| Precision | 0.88 | 0.89 | 0.93 | 0.88 |
| Recall | 0.96 | 0.95 | 0.89 | 0.93 |
| Specificity | 0.86 | 0.87 | 0.93 | 0.86 |
| F1 | 0.92 | 0.91 | 0.91 | 0.91 |

Test-set counts, rows actual and columns predicted:

![Chi-square confusion matrices](figures/chi-square-confusion.png)

![Literature-feature confusion matrices](figures/literature-confusion.png)

Interceptions do most of the work. The tree is the model to show someone, because the splits are readable. KNN matches its accuracy and, on the literature features, trades precision for recall.


## 5. Clusters

Hierarchical clustering uses Ward's method on the scaled numeric columns. The elbow flattens after about four clusters. Average silhouette is highest at two clusters and declines as k grows, so four is a choice about a useful cut rather than a unique optimum. The dendrogram was cut at four.

![Elbow plot](figures/elbow.png)

![Silhouette plot](figures/silhouette.png)


In [ ]:
df_for_clustering <- df[, sapply(df, is.numeric)]
df_scaled <- scale(na.omit(df_for_clustering))

hc <- hclust(dist(df_scaled, method = "euclidean"), method = "ward.D2")
df$cluster <- as.factor(cutree(hc, 4))

# Bootstrap stability of the same hierarchical cut
clusterboot_hc <- clusterboot(
  df_scaled,
  clustermethod = hclustCBI,
  method = "ward.D2",
  k = 4
)


Bootstrap stability (1 minus the dissolution rate) is 0.69, 0.55, 0.09, and 0.53 for clusters 1 to 4. Cluster 3 dissolves. Treat that group as a description of this fit, and lean on clusters 1, 2, and 4 when talking about stable types.

The scatter below is a separate 4-means fit drawn on the first two principal components, so the colours are a k-means partition of the same scaled matrix. The first component explains 38% of the variance and the second 19% (57% together). The bar charts further down use the hierarchical cut, `cutree(hc, 4)`.

![Four k-means groups on the first two principal components](figures/pca-clusters.png)


In [ ]:
pca <- prcomp(df_scaled)
pca_data <- data.frame(pca$x[, 1:3])

set.seed(4009)
kmeans_result <- kmeans(df_scaled, centers = 4)
pca_data$cluster <- kmeans_result$cluster


### What the four hierarchical groups look like

On the position chart, cluster 2 is the defensive group and cluster 3 is the attacking group. Cluster 1 has a defensive majority with a large attacking share. Cluster 4 has an attacking majority. The empty strip on the right of that figure is a spare facet.

![Attacking and defensive players by cluster](figures/position-by-cluster.png)

Cluster 4 is the young group: its age box sits near 20, while the other three sit in the mid-20s. Cluster 1 has the highest overall ratings, centered in the low 70s, and cluster 4 the lowest.

![Age by cluster](figures/age-by-cluster.png)

![Overall rating by cluster](figures/overall-rating.png)

High work rate (orange) is the largest band in every cluster, for both defending and attacking. Clusters 1 and 2 still show a clear Low band. Clusters 3 and 4 are mostly High. Cluster 1 is also the group with the higher pace, passing, dribbling, and physical summaries.

![Work rate by cluster](figures/work-rate.png)

![Pace, passing, dribbling, and physical by cluster](figures/summary-stats.png)

Read together: cluster 1 is the higher-rated mixed group with a defensive lean, cluster 2 is defensive, cluster 4 is younger and lower-rated, and cluster 3 looks attacking but does not survive the bootstrap check.


## 6. Dashboard

The same results are in a Shiny app: single-variable AUC, the tree and KNN under both feature sets, and the cluster charts. A short walkthrough of the dashboard is at <https://youtu.be/tlGLk1NIcFU>.


## 7. Reproduce

1. Download `fifa21_raw_data.csv` from the [Kaggle dataset](https://www.kaggle.com/datasets/yagunnersya/fifa-21-messy-raw-dataset-for-cleaning-exploring) and put it next to the R script.
2. The analysis used R with `tidyverse`, `rpart`, `rpart.plot`, `caret`, `class`, `FSelector`, `pROC`, `ROCR`, `ggplot2`, `factoextra`, and `fpc`.
3. The split and the k-means fit both use `set.seed(4009)`.

### References

The literature feature list was drawn from published FIFA analyses, including:

- Al-Asadi, M. A., and Tasdemir, S. (2021). Empirical comparisons for combining balancing and feature selection strategies for characterizing football players using FIFA video game system. *IEEE Access*, 9, 149266–149286.
- Soto Valero, C. (2017). A Gaussian mixture clustering model for characterizing football players using the EA Sports' FIFA video game system. *Revista Internacional de Ciencias del Deporte*, 13(49), 244–259.
- Yaldo, L., and Shamir, L. (2017). Computational estimation of football player wages. *International Journal of Computer Science in Sport*, 16(1), 18–38.
